Loading Dataset

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Sanskrit NLP/rv_content_devanagari.csv")

# Drop duplicates early
df = df.drop_duplicates(subset=["vedamantra"]).reset_index(drop=True)

print(len(df))

10440


In [ ]:
import re
def preprocess(text):
    # First, replace the punctuation, then split
    cleaned_text = re.sub(r"[।॥]", " ", text)
    words = cleaned_text.strip().split()
    return words

df["tokens"] = df["vedamantra"].apply(preprocess)

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Sanskrit NLP/rv_content_devanagari.csv")

# Drop duplicates early
df = df.drop_duplicates(subset=["vedamantra"]).reset_index(drop=True)

print(len(df))

10440


In [ ]:
import re
def preprocess(text):
    # First, replace the punctuation, then split
    cleaned_text = re.sub(r"[।॥]", " ", text)
    words = cleaned_text.strip().split()
    return words

df["tokens"] = df["vedamantra"].apply(preprocess)

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Sanskrit NLP/rv_content_devanagari.csv")

# Drop duplicates early
df = df.drop_duplicates(subset=["vedamantra"]).reset_index(drop=True)

print(len(df))

10440


In [ ]:
import re
def preprocess(text):
    # First, replace the punctuation, then split
    cleaned_text = re.sub(r"[।॥]", " ", text)
    words = cleaned_text.strip().split()
    return words

df["tokens"] = df["vedamantra"].apply(preprocess)

In [ ]:
def word_to_sequence(word):
    seq = []

    for char in word:
        if char in phoneme_vectors:
            seq.append(phoneme_vectors[char])

    if len(seq) == 0:
        return None

    return torch.tensor(seq, dtype=torch.float)

In [ ]:
def get_embedding(word):

    seq = word_to_sequence(word)

    if seq is None:
        return None

    seq = seq.unsqueeze(0).to(device)   # [1, seq_len, input_dim]

    with torch.no_grad():
        # Check if the current global 'model' is an instance of ContrastiveModelEncoder
        if isinstance(model, ContrastiveModelEncoder):
            emb = model(seq) # Contrastive model does not take lengths
        else:
            lengths = torch.tensor([len(seq)], device=device)
            emb = model(seq, lengths) # Other models take lengths

    return emb.squeeze(0).cpu().numpy()

In [ ]:
word_embedding_cache = {}

def get_embedding_cached(word):

    if word in word_embedding_cache:
        return word_embedding_cache[word]

    emb = get_embedding(word)

    if emb is not None:
        word_embedding_cache[word] = emb

    return emb

In [ ]:
def sentence_embedding(tokens):

    vecs = []

    for w in tokens:
        emb = get_embedding_cached(w)
        if emb is not None:
            vecs.append(emb)

    if len(vecs) == 0:
        return np.zeros(128)

    return np.mean(vecs, axis=0)

Loading the model

triplet model

In [ ]:
import torch.nn as nn
import torch
import torch.nn.functional as F

In [ ]:
# ### Encoder of triplet model
# class TripletModelEncoder(nn.Module):
#     def __init__(self, input_dim,
#                  hidden_dim=256,
#                  layers=2,
#                  proj_dim=128):
#         super().__init__()

#         self.lstm = nn.LSTM(
#             input_dim,
#             hidden_dim,
#             num_layers=layers,
#             batch_first=True,
#             bidirectional=True
#         )

#         self.attn = nn.Linear(hidden_dim * 2, 1)

#         self.proj = nn.Linear(hidden_dim * 2, proj_dim)

#     def forward(self, x, lengths):

#         packed = nn.utils.rnn.pack_padded_sequence(
#             x, lengths.cpu(), batch_first=True, enforce_sorted=False
#         )

#         out, _ = self.lstm(packed)
#         out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)

#         # mask for padded positions
#         mask = torch.arange(out.size(1)).to(lengths.device)[None, :] < lengths[:, None]

#         scores = self.attn(out).squeeze(-1)   # [batch, seq_len]
#         scores = scores.masked_fill(~mask, -1e9)

#         weights = torch.softmax(scores, dim=1)  # [batch, seq_len]
#         pooled = torch.sum(out * weights.unsqueeze(-1), dim=1)

#         return self.proj(pooled)

In [ ]:
# import torch

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# triplet_model = TripletModelEncoder(
#     input_dim=34,
#     hidden_dim=256,
#     layers=2,
#     proj_dim=128
# ).to(device)

# triplet_model.load_state_dict(
#     torch.load("/content/drive/MyDrive/Sanskrit NLP/models/TML_model.pkl", map_location=device)
# )

# triplet_model.eval()
# print("Triplet model loaded.")

## Metric Learning MOdel Encoder

In [ ]:
# class MetricEncoder(nn.Module):
#     def __init__(self, input_dim=34, hidden_dim=256, layers=2, emb_dim=128):
#         super().__init__()

#         self.lstm = nn.LSTM(
#             input_size=input_dim,
#             hidden_size=hidden_dim,
#             num_layers=layers,
#             batch_first=True,
#             bidirectional=True
#         )

#         # because BiLSTM → hidden_dim * 2
#         self.proj = nn.Linear(hidden_dim * 2, emb_dim)

#     def forward(self, x, lengths):

#         packed = nn.utils.rnn.pack_padded_sequence(
#             x, lengths.cpu(), batch_first=True, enforce_sorted=False
#         )

#         _, (h_n, _) = self.lstm(packed)

#         # h_n shape:
#         # (layers * directions, batch, hidden_dim)

#         # last layer forward
#         forward_h = h_n[-2]

#         # last layer backward
#         backward_h = h_n[-1]

#         h = torch.cat((forward_h, backward_h), dim=1)

#         emb = self.proj(h)

#         return emb


In [ ]:
# import torch

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# input_dim = 34 # Define input_dim here
# metric_model = MetricEncoder(
#     input_dim = input_dim,
#     hidden_dim=256,
#     emb_dim=128
# ).to(device)

# metric_model.load_state_dict(
#     torch.load("/content/drive/MyDrive/Sanskrit NLP/models/metric_learning_model .pkl", map_location=device)
# )

# metric_model.eval()
# print("Metric model loaded.")

## Contrasctive Learning MOdel Encoder

In [ ]:
class ContrastiveModelEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, proj_dim=128):
        super().__init__()

        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.attn = nn.Linear(hidden_dim * 2, 1)

        self.projection = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim * 2),
            nn.ReLU(),
            nn.Linear(hidden_dim * 2, proj_dim)
        )

    def forward(self, x):

        h, _ = self.lstm(x)  # [B, T, 2H]

        attn_scores = self.attn(h)
        attn_weights = torch.softmax(attn_scores, dim=1)

        pooled = torch.sum(attn_weights * h, dim=1)

        z = self.projection(pooled)
        z = F.normalize(z, p=2, dim=1)

        return z

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

INPUT_DIM = 34 # Define input_dim here

contrastive_model = ContrastiveModelEncoder(
    input_dim=INPUT_DIM,
    hidden_dim=256,
    proj_dim=128
).to(device)

contrastive_model.load_state_dict(
    torch.load("/content/drive/MyDrive/Sanskrit NLP/models/contrastive_model.pkl", map_location=device)
)

contrastive_model.eval()
print("Contrastive model loaded.")

Contrastive model loaded.


In [ ]:
model = contrastive_model #set which model to work with

In [ ]:
import numpy as np
phoneme_df = pd.read_csv("/content/drive/MyDrive/Sanskrit NLP/sanskrit_phoneme_vectors (1).csv", index_col=0)

phoneme_vectors = {
    row.Index: np.array(row[1:], dtype=float)
    for row in phoneme_df.itertuples()
}

print(f"Loaded {len(phoneme_vectors)} Sanskrit phonemes with 34 features.")

Loaded 52 Sanskrit phonemes with 34 features.


word to phoneme featues sequence

In [ ]:
def word_to_sequence(word):
    seq = []

    for char in word:
        if char in phoneme_vectors:
            seq.append(phoneme_vectors[char])

    if len(seq) == 0:
        return None

    return torch.tensor(seq, dtype=torch.float)

word embeddings using encoder

In [ ]:
def get_embedding(word):

    seq = word_to_sequence(word)

    if seq is None:
        return None

    seq = seq.unsqueeze(0).to(device)   # [1, seq_len, input_dim]

    with torch.no_grad():
        # Check if the current global 'model' is an instance of ContrastiveModelEncoder
        if isinstance(model, ContrastiveModelEncoder):
            emb = model(seq) # Contrastive model does not take lengths
        else:
            lengths = torch.tensor([len(seq)], device=device)
            emb = model(seq, lengths) # Other models take lengths

    return emb.squeeze(0).cpu().numpy()

In [ ]:
word_embedding_cache = {}

def get_embedding_cached(word):

    if word in word_embedding_cache:
        return word_embedding_cache[word]

    emb = get_embedding(word)

    if emb is not None:
        word_embedding_cache[word] = emb

    return emb

sentence embedding

In [ ]:
def sentence_embedding(tokens):

    vecs = []

    for w in tokens:
        emb = get_embedding_cached(w)
        if emb is not None:
            vecs.append(emb)

    if len(vecs) == 0:
        return np.zeros(128)

    return np.mean(vecs, axis=0)


compute sentence embedings

In [ ]:
import tqdm
embeddings = []

for tokens in tqdm.tqdm(df["tokens"]):
    embeddings.append(sentence_embedding(tokens))

df["embedding"] = embeddings
display(df.head())

100%|██████████| 10440/10440 [00:00<00:00, 23441.74it/s]


,vedamantra,chanda,tokens,embedding
0,अग्निमीले पुरोहितं यज्ञस्य देवमृत्विजम् । होता...,गायत्री,"[अग्निमीले, पुरोहितं, यज्ञस्य, देवमृत्विजम्, ह...","[-0.0037692655, -0.041660357, -0.0023088194, 0..."
1,अग्निः पूर्वेभिरृषिभिरीड्यो नूतनैरुत । स देवाम...,गायत्री,"[अग्निः, पूर्वेभिरृषिभिरीड्यो, नूतनैरुत, स, दे...","[0.00720469, -0.0084880795, 0.037656356, 0.043..."
2,अग्निना रयिमश्नवत्पोषमेव दिवेदिवे । यशसं वीरवत...,गायत्री,"[अग्निना, रयिमश्नवत्पोषमेव, दिवेदिवे, यशसं, वी...","[0.0032672752, -0.037255194, 0.05016138, -0.01..."
3,अग्ने यं यज्ञमध्वरं विश्वतः परिभूरसि । स इद्दे...,गायत्री,"[अग्ने, यं, यज्ञमध्वरं, विश्वतः, परिभूरसि, स, ...","[-0.0011298029, -0.02514022, -0.0013834946, 0...."
4,अग्निर्होता कविक्रतुः सत्यश्चित्रश्रवस्तमः । द...,गायत्री,"[अग्निर्होता, कविक्रतुः, सत्यश्चित्रश्रवस्तमः,...","[0.044388846, -0.06500394, 0.010963344, 0.0665..."


normalize the embeddings

In [ ]:
df["embedding"] = [
    emb / np.linalg.norm(emb) if np.linalg.norm(emb) != 0 else emb
    for emb in df["embedding"]
]

In [ ]:
df["embedding"] = df["embedding"].apply(lambda x: x.tolist())

In [ ]:
display(df.head())

,vedamantra,chanda,tokens,embedding
0,अग्निमीले पुरोहितं यज्ञस्य देवमृत्विजम् । होता...,गायत्री,"[अग्निमीले, पुरोहितं, यज्ञस्य, देवमृत्विजम्, ह...","[-0.0037692654877901077, -0.041660357266664505..."
1,अग्निः पूर्वेभिरृषिभिरीड्यो नूतनैरुत । स देवाम...,गायत्री,"[अग्निः, पूर्वेभिरृषिभिरीड्यो, नूतनैरुत, स, दे...","[0.007204690016806126, -0.008488079532980919, ..."
2,अग्निना रयिमश्नवत्पोषमेव दिवेदिवे । यशसं वीरवत...,गायत्री,"[अग्निना, रयिमश्नवत्पोषमेव, दिवेदिवे, यशसं, वी...","[0.003267275169491768, -0.037255194038152695, ..."
3,अग्ने यं यज्ञमध्वरं विश्वतः परिभूरसि । स इद्दे...,गायत्री,"[अग्ने, यं, यज्ञमध्वरं, विश्वतः, परिभूरसि, स, ...","[-0.00112980289850384, -0.025140220299363136, ..."
4,अग्निर्होता कविक्रतुः सत्यश्चित्रश्रवस्तमः । द...,गायत्री,"[अग्निर्होता, कविक्रतुः, सत्यश्चित्रश्रवस्तमः,...","[0.044388845562934875, -0.06500393897294998, 0..."


In [ ]:
df.to_csv("normalized_embeddings.csv", index=False, encoding = 'utf-8-sig')

In [ ]:
print(f"Length of one sentence embedding: {len(df['embedding'].iloc[0])}")


Length of one sentence embedding: 128


Getting top k similar

In [ ]:
# from sklearn.metrics.pairwise import euclidean_distances

# def get_top_k_similar(query_idx, k=5):

#     query_vec = df.loc[query_idx, "embedding"].reshape(1, -1)
#     query_chanda = df.loc[query_idx, "chanda"]
#     query_text = df.loc[query_idx, "vedamantra"]

#     # Filter same chanda
#     filtered_df = df[df["chanda"] == query_chanda].copy()

#     # Remove same mantra
#     filtered_df = filtered_df[filtered_df["vedamantra"] != query_text]

#     if len(filtered_df) == 0:
#         return None

#     mat = np.vstack(filtered_df["embedding"].values)

#     # Compute Euclidean distance
#     dists = euclidean_distances(query_vec, mat)[0]

#     filtered_df["distance"] = dists

#     # Sort ASCENDING (smaller distance = more similar)
#     results = filtered_df.sort_values(by="distance", ascending=True).head(k)

#     return results[["vedamantra", "chanda", "distance"]]

In [ ]:
# query_idx = 10

# print("Query Mantra:\n", df.loc[query_idx, "vedamantra"])

# results = get_top_k_similar(query_idx, k=5)

# print("\nTop-K Similar Mantras:\n")
# print(results)

In [ ]:
def get_query_embedding(text):

    tokens = preprocess(text)
    emb = sentence_embedding(tokens)

    # normalize (VERY IMPORTANT)
    emb = emb / np.linalg.norm(emb)

    return emb.reshape(1, -1)

In [ ]:
from sklearn.metrics.pairwise import euclidean_distances, cosine_similarity


def retrieve_similar_mantras(query_text, k=5, chanda=None):

    query_vec = get_query_embedding(query_text)

    # Optional: filter by chanda
    if chanda is not None:
        filtered_df = df[df["chanda"] == chanda].copy()
    else:
        filtered_df = df.copy()

    # Remove exact match (avoid redundancy)
    filtered_df = filtered_df[filtered_df["vedamantra"] != query_text]

    if len(filtered_df) == 0:
        return None

    mat = np.vstack(filtered_df["embedding"].values)

    sims = cosine_similarity(query_vec, mat)[0]

    filtered_df["similarity"] = sims

    results = filtered_df.sort_values(by="similarity", ascending=False).head(k)

    return results[["vedamantra", "chanda", "similarity"]]

In [ ]:
query_text = "नू ष्टुत इन्द्र नू गृणान इषं जरित्रे नद्यो3 न पीपेः । अकारि ते हरिवो ब्रह्म नव्यं धिया स्याम रथ्यः सदासाः ॥"

results = retrieve_similar_mantras(query_text, k=100)

print("\nQuery:\n", query_text)
print("\nTop-K Similar Mantras:\n")
print(results)


Query:
 नू ष्टुत इन्द्र नू गृणान इषं जरित्रे नद्यो3 न पीपेः । अकारि ते हरिवो ब्रह्म नव्यं धिया स्याम रथ्यः सदासाः ॥

Top-K Similar Mantras:

                                             vedamantra          chanda  \
7002  न ते गिरो अपि मृष्ये तुरस्य न सुष्टुतिमसुर्यस्...          विराट्   
7963  येषामर्णो न सप्रथो नाम त्वेषं शश्वतामेकमिद्भुज...  ककुप् -उष्णिक्   
6564  नम इदुग्रं नम आ विवासे नमो दाधार पृथिवीमुत द्य...      त्रिष्टुप्   
1010  सूर्ये विषमा सजामि दृतिं सुरावतो गृहे । सो चिन...     महापङ्क्तिः   
2822  एते शमीभिः सुशमी अभूवन्ये हिन्विरे तन्वः1 सोम ...      त्रिष्टुप्   
...                                                 ...             ...   
3397  ग्रावाणो न सूरयः सिन्धुमातर आदर्दिरासो अद्रयो ...            जगती   
7820  सत्रा त्वं पुरुष्टुतम् एको वृत्राणि तोशसे । ना...         उष्णिक्   
6421  या त ऊतिरमित्रहन्मक्षूजवस्तमासति । तया नो हिनु...         गायत्री   
9281  उक्थवाहसे विभ्वे मनीषां द्रुणा न पारमीरया नदीन...      त्रिष्टुप्   
1795  इन्द्रो मदाय वावृधे शवसे वृ

In [ ]:
# -*- coding: utf-8 -*-

INDEPENDENT_VOWELS = set([
    'अ','आ','इ','ई','उ','ऊ','ऋ','ॠ','ऌ','ए','ऐ','ओ','औ'
])

VOWEL_SIGNS = set([
    'ा','ि','ी','ु','ू','ृ','ॄ','ॢ','े','ै','ो','ौ'
])

HALANT = '्'

CONSONANTS = set([
    'क','ख','ग','घ','ङ',
    'च','छ','ज','झ','ञ',
    'ट','ठ','ड','ढ','ण',
    'त','थ','द','ध','न',
    'प','फ','ब','भ','म',
    'य','र','ल','व',
    'श','ष','स','ह'
])


def extract_units(word):
    units = []
    i = 0

    while i < len(word):
        ch = word[i]

        # Independent vowel
        if ch in INDEPENDENT_VOWELS:
            units.append((ch, 'V'))
            i += 1

        # Consonant
        elif ch in CONSONANTS:
            token = ch
            has_vowel = False
            i += 1

            # Halant → dead consonant
            if i < len(word) and word[i] == HALANT:
                token += HALANT
                units.append((token, 'C'))
                i += 1

            # Matra → CV unit
            elif i < len(word) and word[i] in VOWEL_SIGNS:
                token += word[i]
                has_vowel = True
                units.append((token, 'CV'))
                i += 1

            else:
                # inherent 'अ'
                units.append((token, 'CV'))

        else:
            units.append((ch, 'O'))  # others (anusvara, visarga etc.)
            i += 1

    return units

In [ ]:
print(extract_units("भारत"))

[('भा', 'CV'), ('र', 'CV'), ('त', 'CV')]


In [ ]:
def syllabify_from_units(units):
    syllables = []
    i = 0

    while i < len(units):
        syllable = []

        # Step 1: must start with CV or V
        syllable.append(units[i][0])

        if units[i][1] not in ['CV', 'V']:
            i += 1
            continue

        i += 1

        # Step 2: collect following consonants
        cluster = []
        j = i

        while j < len(units) and units[j][1] == 'C':
            cluster.append(units[j][0])
            j += 1

        # Apply rules
        if len(cluster) == 0:
            pass

        elif len(cluster) == 1:
            # goes to next syllable
            pass

        else:
            # split after first consonant
            syllable.append(cluster[0])
            i += 1

        syllables.append("".join(syllable))

    return syllables


def count_syllables(word):
    units = extract_units(word)
    syllables = syllabify_from_units(units)
    return syllables, len(syllables)

In [ ]:
words = [
    "भारत",
    "शकुन्तला",
    "कृत्स्नम्",
    "अकर",
    "धर्म",
    "गच्छति",
    "अग्निमीले"

]

for w in words:
    syl, count = count_syllables(w)
    print(f"{w} → {syl} → {count}")

भारत → ['भा', 'र', 'त'] → 3
शकुन्तला → ['श', 'कु', 'त', 'ला'] → 4
कृत्स्नम् → ['कृत्', 'न'] → 2
अकर → ['अ', 'क', 'र'] → 3
धर्म → ['ध', 'म'] → 2
गच्छति → ['ग', 'छ', 'ति'] → 3
अग्निमीले → ['अ', 'नि', 'मी', 'ले'] → 4


In [ ]:
भारत → ['भा', 'र', 'त'] → 3
शकुन्तला → ['श', 'कुन्', 'त', 'ला'] → 4
कृत्स्नम् → ['कृत्', 'स्नम्'] → 2
अकर → ['अ', 'क', 'र'] → 3
धर्म → ['ध', 'र्म'] → 2
गच्छति → ['ग', 'च्छ', 'ति'] → 3